# NB02 — Classical vs Quantum Complexity Crossover

**Research question:** At what qubit count $Q^*$ does classical simulation become infeasible,
and quantum hardware provides a genuine computational advantage?

## What this notebook proves
- **Section A:** Classical analytical kernel time scales polynomially in $Q$ — but statevector simulation scales as $O(2^Q)$
- **Section B:** Quantum circuit gate count for $K_{ZZ}$ scales as $O(Q^2)$ — much better than $O(2^Q)$
- **Section C:** Crossover plot — find $Q^*$ for three hardware generations (2024, 2027, 2032)

Key takeaway: our analytical kernels (numpy) are always faster for $Q \leq 20$.
Real quantum advantage appears for $Q \geq 25$–30 with near-term hardware.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import time
import matplotlib.pyplot as plt
from src.kernels.analytical import K_ZZ
from src.kernels.diagnostics import time_kernel_fn, count_circuit_resources

plt.rcParams.update({'font.family': 'sans-serif', 'figure.dpi': 130})
print('Imports OK')

## Section A — Classical Wall: Timing the Analytical Kernel

Our `K_ZZ` is a closed-form numpy formula with $O(Q^2 \cdot N^2)$ complexity.
We time it for $N=100$ samples and increasing $Q$, then fit a regression to extrapolate.

The **statevector simulation** alternative (Qiskit) needs $O(2^Q)$ memory and time per circuit —
this is the wall that makes classical simulation infeasible beyond $Q \approx 25$.

In [ ]:
Q_range_A = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
N_SAMPLES = 100
N_REPEATS = 5

times_analytical = []

for Q in Q_range_A:
    np.random.seed(0)
    X = np.random.rand(N_SAMPLES, Q) * 2  # scaled to [0,2] like QuantumScaler
    t = time_kernel_fn(K_ZZ, X, alpha=1.0, n_repeats=N_REPEATS)
    times_analytical.append(t)
    print(f'  Q={Q:2d}: {t*1000:.2f} ms')

times_analytical = np.array(times_analytical)
Q_arr_A = np.array(Q_range_A)

In [ ]:
# Fit log(T) = a + b*Q (analytical kernel grows polynomially, not exponentially)
log_t = np.log(times_analytical)
coeffs_time = np.polyfit(Q_arr_A, log_t, 1)

Q_extrap = np.array([2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 25, 30, 40, 50, 100])
log_t_extrap = np.polyval(coeffs_time, Q_extrap)
t_analytical_extrap = np.exp(log_t_extrap)

# Statevector simulation: O(N^2 * 2^Q) time, 1ns per amplitude operation
# N*(N-1)/2 pairs, each needs 2 * 2^Q operations (fidelity = |<ψ|φ>|²)
T_sv_extrap = (N_SAMPLES * (N_SAMPLES - 1) / 2) * 2 * (2.0 ** Q_extrap) * 1e-9

print(f'Analytical fit: log(T) = {coeffs_time[0]:.4f}*Q + {coeffs_time[1]:.4f}')
print(f'Doubling every {np.log(2)/coeffs_time[0]:.1f} qubits (analytical)')
print()
print('Extrapolation (analytical):')
for Q, t in zip(Q_extrap, t_analytical_extrap):
    if Q > 20:
        print(f'  Q={Q:3d}: {t:.2e} s (analytical), {T_sv_extrap[Q_extrap==Q][0]:.2e} s (statevector)')

## Section B — Quantum Circuit Resources

A `ZZFeatureMap(Q, reps=1)` circuit has:
- Single-qubit gates: $O(Q)$  
- CNOT gates: $O(Q^2)$ (all pairs interact)

The **compute-uncompute** circuit for kernel evaluation = 2 × feature map → total gates $\sim O(Q^2)$.

We verify this empirically with `count_circuit_resources`, with analytical fallback if Qiskit unavailable.

In [ ]:
Q_range_B = [2, 4, 6, 8, 10, 12, 16, 20, 30, 50]

resources = []
for Q in Q_range_B:
    r = count_circuit_resources(Q, 'ZZ', reps=1)
    resources.append(r)
    print(f'  Q={Q:2d}: CX={r["n_cnot"]:4d}, 1q={r["n_single"]:4d}, '
          f'total={r["total_gates"]:4d}, depth={r["depth"]:3d}')

In [ ]:
# Kernel circuit = ZZFeatureMap + inverse → 2× gates
Q_res = np.array([r['Q'] for r in resources])
total_gates_fm = np.array([r['total_gates'] for r in resources])
total_gates_kernel = 2 * total_gates_fm  # compute-uncompute
depths_fm = np.array([r['depth'] for r in resources])
depths_kernel = 2 * depths_fm

# Fit: log(gates) vs log(Q) to determine scaling exponent
log_Q = np.log(Q_res.astype(float))
log_gates = np.log(total_gates_kernel.astype(float))
coeffs_gates = np.polyfit(log_Q, log_gates, 1)
exponent = coeffs_gates[0]

Q_fit_B = np.linspace(2, 60, 200)
gates_fit = np.exp(np.polyval(coeffs_gates, np.log(Q_fit_B)))

print(f'Gate count scaling: O(Q^{exponent:.2f}) (expected O(Q^2) = O(Q^2.00))')
print()
print('Kernel circuit total gates (compute-uncompute):')
for Q, g, d in zip(Q_res, total_gates_kernel, depths_kernel):
    print(f'  Q={Q:2d}: {g} gates, depth {d}')

## Section C — Crossover Plot

We compare three time estimates for evaluating an $N \times N$ kernel matrix:

1. **Classical analytical** (numpy `K_ZZ`): measured + extrapolated
2. **Classical statevector** (Qiskit simulator): $O(N^2 \cdot 2^Q)$
3. **Quantum hardware** (3 scenarios): $T = N(N-1)/2 \times \text{shots} \times (\text{depth} \times t_{CX} + t_{readout})$

Hardware scenarios:
- **IBM Torino (2024)**: $t_{CX}=60$ns, shots=1024
- **Near-term (2027)**: $t_{CX}=10$ns, shots=256
- **Future (2032)**: $t_{CX}=1$ns, shots=64

In [ ]:
# Hardware parameters
hw_configs = [
    {'name': 'IBM Torino (2024)', 'color': '#e74c3c',
     't_cx': 60e-9, 't_single': 30e-9, 't_readout': 1.2e-6, 'shots': 1024},
    {'name': 'Near-term (2027)',  'color': '#f39c12',
     't_cx': 10e-9, 't_single': 5e-9,  't_readout': 200e-9, 'shots': 256},
    {'name': 'Future (2032)',     'color': '#2ecc71',
     't_cx': 1e-9,  't_single': 0.5e-9, 't_readout': 50e-9, 'shots': 64},
]

# Q values for crossover analysis


In [ ]:
Q_cross = np.array([2, 4, 6, 8, 10, 12, 14, 16, 20, 25, 30, 40, 50, 100])

# Classical analytical time (extrapolated from fit)
T_analytical = np.exp(np.polyval(coeffs_time, Q_cross))

# Classical statevector: O(N^2 * 2^Q), 1ns per amp
N = 100
T_statevec = (N * (N - 1) / 2) * 2.0 * (2.0 ** Q_cross.astype(float)) * 1e-9
# Cap at 1 year to avoid inf in plots
T_statevec = np.clip(T_statevec, 1e-9, 3.15e7)

# Quantum hardware times
# Depth of kernel circuit (compute-uncompute) at each Q:
# Use analytical formula: depth = 2 * reps * (Q + 1)
depth_kernel_Q = 2 * 1 * (Q_cross + 1)  # reps=1
n_pairs = N * (N - 1) / 2

T_quantum = {}
for cfg in hw_configs:
    T_hw = n_pairs * cfg['shots'] * (depth_kernel_Q * cfg['t_cx'] + cfg['t_readout'])
    T_quantum[cfg['name']] = T_hw
    print(f"{cfg['name']}:")
    for Q, t in zip(Q_cross, T_hw):
        print(f'  Q={Q:3d}: {t:.2e} s')

In [ ]:
# Find crossover points Q* where quantum < classical statevector
print('=== Crossover Analysis ===')
print('Q* where quantum hardware is faster than classical STATEVECTOR simulation:')
print()
for cfg in hw_configs:
    T_hw = T_quantum[cfg['name']]
    ratio = T_statevec / T_hw
    # Find first Q where quantum is faster
    faster_mask = ratio > 1.0
    if faster_mask.any():
        q_star = Q_cross[faster_mask][0]
        print(f"  {cfg['name']}: Q* ≈ {q_star} qubits")
    else:
        print(f"  {cfg['name']}: quantum still slower at Q=100")

print()
print('Note: Classical ANALYTICAL kernel is always faster than statevector simulation.')
print('For quantum advantage vs analytical, need specialized quantum hardware.')

In [ ]:
# ── Key table ─────────────────────────────────────────────────────────────────
print('\n=== Time Table (N=100 samples, N×N kernel matrix) ===')
print(f'{"Q":>6} | {"Classical (analytical)":>24} | {"Statevec sim":>16} | '
      f'{"IBM Torino":>14} | {"Near-term":>14} | {"Future":>14} | Winner')
print('-' * 115)

for i, Q in enumerate([10, 20, 30, 50, 100]):
    idx = np.where(Q_cross == Q)[0]
    if len(idx) == 0:
        continue
    idx = idx[0]
    t_an = T_analytical[idx]
    t_sv = T_statevec[idx]
    times_dict = {'Analytical': t_an, 'Statevec': t_sv}
    for cfg in hw_configs:
        times_dict[cfg['name']] = T_quantum[cfg['name']][idx]
    winner = min(times_dict, key=times_dict.get)
    t_hw = [T_quantum[cfg['name']][idx] for cfg in hw_configs]
    print(f'{Q:6d} | {t_an:>24.2e} | {t_sv:>16.2e} | '
          f'{t_hw[0]:>14.2e} | {t_hw[1]:>14.2e} | {t_hw[2]:>14.2e} | {winner}')

In [ ]:
import os
os.makedirs('../results/figures', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Classical vs Quantum Complexity Crossover', fontsize=13, fontweight='bold')

# ── Panel A: Classical timing ─────────────────────────────────────────────────
ax = axes[0]
ax.semilogy(Q_arr_A, times_analytical * 1000, 'o-', color='#3498db',
            label='Measured (K_ZZ numpy)', zorder=5)
ax.semilogy(Q_extrap, t_analytical_extrap * 1000, '--', color='#2980b9',
            alpha=0.7, label=f'Fit (slope={coeffs_time[0]:.3f}/qubit)')
ax.semilogy(Q_extrap, T_sv_extrap * 1000, '-.', color='#e74c3c',
            label='Statevector sim O(2^Q)')
ax.set_xlabel('Number of qubits Q')
ax.set_ylabel('Time (ms) for N=100 samples')
ax.set_title('(A) Classical Kernel Timing')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim([2, 50])

# ── Panel B: Circuit resources ────────────────────────────────────────────────


In [ ]:
ax = axes[1]
ax.loglog(Q_res, total_gates_kernel, 'D-', color='#9b59b6',
          label='Total gates (kernel circuit)', zorder=5)
ax.loglog(Q_fit_B, gates_fit, '--', color='#8e44ad', alpha=0.7,
          label=f'Fit: O(Q^{exponent:.2f})')
# Reference Q^2 line
ax.loglog(Q_fit_B, 3 * Q_fit_B**2, ':', color='gray', alpha=0.6, label='O(Q²) reference')
ax.set_xlabel('Number of qubits Q')
ax.set_ylabel('Total gates (compute-uncompute)')
ax.set_title('(B) Quantum Circuit Resources')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Panel C: Full crossover plot ──────────────────────────────────────────────


In [ ]:
ax = axes[2]

# Classical lines
ax.loglog(Q_cross, T_analytical, '-', color='#3498db', lw=2, label='Classical analytical')
ax.loglog(Q_cross, T_statevec, '-', color='#e74c3c', lw=2, label='Statevector sim')

# Hardware lines
for cfg in hw_configs:
    T_hw = T_quantum[cfg['name']]
    ax.loglog(Q_cross, T_hw, '--', color=cfg['color'], lw=1.8, label=cfg['name'])



In [ ]:
# Shade quantum advantage region (where quantum < statevector)
T_best_quantum = np.minimum.reduce([T_quantum[cfg['name']] for cfg in hw_configs])
advantage_mask = T_best_quantum < T_statevec
if advantage_mask.any():
    q_adv_start = Q_cross[advantage_mask][0]
    ax.axvspan(q_adv_start, Q_cross[-1], alpha=0.08, color='#27ae60',
               label=f'Quantum adv. vs statevec (Q≥{q_adv_start})')

ax.set_xlabel('Number of qubits Q')
ax.set_ylabel('Wall-clock time (s), N=100 samples')
ax.set_title('(C) Crossover: Classical vs Quantum')
ax.legend(fontsize=7, loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim([2, 100])
ax.set_ylim([1e-6, 1e8])

# Add 1 year, 1 day, 1 hour reference lines
for label, t_ref in [('1 year', 3.15e7), ('1 day', 86400), ('1 hour', 3600)]:
    ax.axhline(t_ref, color='gray', ls=':', lw=0.8, alpha=0.5)
    ax.text(3, t_ref * 1.2, label, fontsize=7, color='gray')

plt.tight_layout()
save_path = '../results/figures/NB02_complexity_crossover.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Figure saved to {save_path}')
plt.show()